<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/Half_Kelly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import gc
import logging
import time
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from sklearn.utils.class_weight import compute_sample_weight

# 關閉 yfinance 底層過多警告訊號
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

# 1. 環境相容性設定 (Colab 自動下載與本地執行相容)
try:
    from google.colab import files

    HAS_COLAB = True
except ImportError:
    HAS_COLAB = False

try:
    from IPython.display import display
except ImportError:

    def display(df):
        print(df.to_string())


# 設定預測信心門檻
CONFIDENCE_THRESHOLD = 0.40

# ==============================================================================
# 0. 股票資料池定義 (完整 160 隻標的 - 完全保留)
# ==============================================================================
stock_dict = {
    "0050.TW": "元大台灣50",
    "0056.TW": "元大高股息",
    "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式",
    "SPCX": "SPACs ETF",
    "SOXX": "iShares半導體ETF",
    "SMH": "VanEck半導體ETF",
    "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet",
    "META": "Meta",
    "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達",
    "TSM": "台積電 ADR",
    "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐",
    "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生",
    "ASML": "ASML 艾司摩爾",
    "AMAT": "Applied Materials 應用材料",
    "LRCX": "Lam Research 柯林研發",
    "KLAC": "KLA 科磊",
    "AMD": "AMD 超微",
    "AVGO": "Broadcom 博通",
    "QCOM": "Qualcomm 高通",
    "INTC": "Intel 英特爾",
    "MU": "Micron 鎂光",
    "TXN": "Texas Instruments 德州儀器",
    "ARM": "ARM 晶心/安謀",
    "MRVL": "Marvell 邁威爾",
    "ADI": "Analog Devices 亞德諾",
    "MPWR": "Monolithic Power 芯源系統",
    "ON": "ON Semiconductor 安森美",
    "SWKS": "Skyworks 思佳訊",
    "QRVO": "Qorvo 威訊",
    "TER": "Teradyne 泰瑞達",
    "MKSI": "MKS Instruments",
    "PANW": "Palo Alto Networks",
    "CRWD": "CrowdStrike",
    "FTNT": "Fortinet",
    "NET": "Cloudflare",
    "ZS": "Zscaler",
    "OKTA": "Okta",
    "S": "SentinelOne",
    "GEN": "Gen Digital",
    "RPD": "Rapid7",
    "CBRS": "CyberArk",
    "2471.TW": "資通",
    "2480.TW": "敦陽科",
    "3029.TW": "零壹",
    "6214.TW": "精誠",
    "3130.TW": "一零四",
    "2427.TW": "三商電",
    "3027.TW": "盛達",
    "5203.TW": "訊連",
    "5471.TW": "松翰",
    "5410.TW": "國統",
    "6183.TW": "關貿",
    "6203.TWO": "海韻電",
    "6210.TWO": "慶生",
    "6593.TWO": "台灣銘板",
    "6689.TW": "伊雲谷",
    "6690.TWO": "安碁資訊",
    "6752.TWO": "睿嘉",
    "6763.TWO": "綠界科技",
    "6865.TWO": "偉康科技",
    "6874.TWO": "倍力",
    "6928.TW": "全達",
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微 MSI",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "2353.TW": "宏碁",
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    "3035.TW": "智原",
    "6643.TWO": "M31",
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創唯",
    "6756.TW": "威鋒電子",
    "2342.TW": "茂矽",
    "6770.TW": "力積電",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "钛昇",
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    "2404.TW": "漢唐",
    "1773.TW": "勝一",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "5536.TWO": "聖暉*",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜晶",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    "6715.TW": "嘉基",
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    "8043.TWO": "蜜望實",
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    "2395.TW": "研華",
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    "3481.TW": "群創",
    "2409.TW": "友達",
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "2393.TW": "億光",
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    "2603.TW": "長榮",
    "2609.TW": "陽明",
    "2615.TW": "萬海",
    "2605.TW": "新興",
    "2606.TW": "裕民",
    "2612.TW": "中航",
    "2617.TW": "台航",
    "2637.TW": "慧洋-KY",
    "2641.TWO": "正德",
    "5608.TW": "四維航",
    "2610.TW": "華航",
    "2618.TW": "長榮航",
    "2630.TW": "亞航",
    "5603.TWO": "陸海",
    "2607.TW": "勞運",
    "2608.TW": "嘉里大榮",
    "2611.TW": "志信",
    "2613.TW": "中櫃",
    "2636.TW": "台驊投控",
    "2642.TW": "宅配通",
    "2633.TW": "台灣高鐵",
    "5607.TW": "遠雄港",
    "5609.TWO": "中菲行",
    "8367.TW": "建新國際",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "1303.TW": "南亞",
    "2465.TW": "麗臺",
    "8163.TW": "達方",
    "3042.TW": "晶技",
    "8182.TWO": "加高",
    "3229.TW": "泰藝",
    "3308.TW": "聯傑",
    "6284.TW": "佳邦",
    "2484.TW": "希華",
    "8088.TWO": "華信科",
}


# ==============================================================================
# 1. 大盤環境動態監測 (Macro Market Regimes)
# ==============================================================================
def get_macro_market_status():
    """動態判斷台/美大盤環境（強勢多頭、中性多頭、震盪整理、保守空頭）"""
    try:
        tw_index = yf.download(
            "^TWII", period="1y", interval="1d", progress=False
        )
        if isinstance(tw_index.columns, pd.MultiIndex):
            tw_index.columns = tw_index.columns.get_level_values(0)

        close = tw_index["Close"].iloc[-1]
        ma20 = tw_index["Close"].rolling(20).mean().iloc[-1]
        ma60 = tw_index["Close"].rolling(60).mean().iloc[-1]

        if close > ma20 and ma20 > ma60:
            return "強勢多頭", 1.2
        elif close > ma60:
            return "中性多頭", 1.0
        elif close > ma20:
            return "震盪整理", 0.7
        else:
            return "保守空頭", 0.3
    except Exception:
        return "中性多頭", 1.0


# ==============================================================================
# 2. 數據獲取模組 (三大法人籌碼與 K 線)
# ==============================================================================
def get_tw_chip_data(stock_id):
    """取得台股三大法人買賣超資料 (FinMind API，防範 Rate Limit 並加入延遲防護)"""
    clean_id = stock_id.split(".")[0]
    url = f"https://api.finmindtrade.com/api/v4/data?dataset=TaiwanStockInstitutionalInvestorsBuySell&data_id={clean_id}&start_date=2024-01-01"
    try:
        time.sleep(0.1)  # 加入 100ms 請求間隔防禦 Rate Limit
        res = requests.get(url, timeout=4).json()
        if res.get("status") == 200 and len(res.get("data", [])) > 0:
            df_chip = pd.DataFrame(res["data"])
            df_chip["date"] = pd.to_datetime(df_chip["date"])
            foreign_chip = df_chip[
                df_chip["name"].str.contains("Foreign", na=False)
            ]
            chip_pivot = (
                foreign_chip.groupby("date")["buy"].sum()
                - foreign_chip.groupby("date")["sell"].sum()
            )
            return chip_pivot
    except Exception:
        pass
    return pd.Series(dtype=float)


def fetch_stock_data(ticker):
    """下載股價並進行特徵工程計算"""
    try:
        df = yf.download(ticker, period="3y", interval="1d", progress=False)
        if df.empty or len(df) < 120:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 基礎指標計算
        df["MA5"] = df["Close"].rolling(5).mean()
        df["MA20"] = df["Close"].rolling(20).mean()
        df["MA60"] = df["Close"].rolling(60).mean()

        # RSI (14)
        delta = df["Close"].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / (loss + 1e-9)
        df["RSI"] = 100 - (100 / (1 + rs))

        # MACD
        exp1 = df["Close"].ewm(span=12, adjust=False).mean()
        exp2 = df["Close"].ewm(span=26, adjust=False).mean()
        df["MACD"] = exp1 - exp2
        df["Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

        # ATR & 波動度
        tr = np.maximum(
            df["High"] - df["Low"],
            np.maximum(
                abs(df["High"] - df["Close"].shift(1)),
                abs(df["Low"] - df["Close"].shift(1)),
            ),
        )
        df["ATR"] = tr.rolling(14).mean()

        # 籌碼面/量能特徵
        if ".TW" in ticker or ".TWO" in ticker:
            chip_series = get_tw_chip_data(ticker)
            df["Chip_Net"] = chip_series.reindex(df.index).fillna(0)
            df["Foreign_Ratio"] = df["Chip_Net"] / (df["Volume"] + 1e-9)
        else:
            df["Foreign_Ratio"] = (
                df["Volume"] - df["Volume"].rolling(5).mean()
            ) / (df["Volume"].rolling(5).mean() + 1e-9)

        # Target: 未來 5 日回報率是否 > 3%
        df["Future_Return"] = df["Close"].shift(-5) / df["Close"] - 1.0
        df["Target"] = (df["Future_Return"] > 0.03).astype(int)

        df.dropna(
            subset=["MA60", "RSI", "MACD", "ATR", "Foreign_Ratio"], inplace=True
        )
        return df
    except Exception:
        return None


# ==============================================================================
# 3. LightGBM 模型訓練與扣除交易成本之 EV 評估
# ==============================================================================
def train_and_predict(ticker, df):
    """訓練機器學習模型並產出預測數據（考慮扣除實務交易成本）"""
    features = [
        "MA5",
        "MA20",
        "MA60",
        "RSI",
        "MACD",
        "Signal",
        "ATR",
        "Foreign_Ratio",
    ]
    X = df[features]
    y = df["Target"]

    train_size = int(len(df) * 0.8)
    if train_size < 60:
        return None

    X_train, y_train = X.iloc[:train_size], y.iloc[:train_size]
    X_test, y_test = X.iloc[train_size:-5], y.iloc[train_size:-5]

    if len(y_train.unique()) < 2:
        return None

    weights = compute_sample_weight("balanced", y_train)
    model = lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.03,
        max_depth=4,
        num_leaves=15,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train, sample_weight=weights)

    # 歷史回測與 EV 計算（扣除 0.4% 的手續費與證交稅成本）
    COST = 0.004
    test_preds = model.predict_proba(X_test)[:, 1]
    test_signals = test_preds > CONFIDENCE_THRESHOLD

    actual_returns = (
        df["Future_Return"].iloc[train_size:-5][test_signals] - COST
    )
    hist_win_rate = (
        (actual_returns > 0).mean() if len(actual_returns) > 0 else 0.35
    )

    avg_win = (
        actual_returns[actual_returns > 0].mean()
        if len(actual_returns[actual_returns > 0]) > 0
        else 0.05
    )
    avg_loss = (
        abs(actual_returns[actual_returns < 0].mean())
        if len(actual_returns[actual_returns < 0]) > 0
        else 0.03
    )

    # 單筆期望值 EV
    ev = (hist_win_rate * avg_win) - ((1 - hist_win_rate) * avg_loss)

    # 最新一天 (T+0) 預測機率
    latest_X = X.iloc[[-1]]
    latest_pred_prob = model.predict_proba(latest_X)[0][1]

    # 近期籌碼動向
    recent_chip = df["Foreign_Ratio"].iloc[-1]

    return {
        "p_pred": latest_pred_prob,
        "p_hist": hist_win_rate,
        "ev": ev,
        "avg_win": avg_win,
        "avg_loss": avg_loss,
        "chip_ratio": recent_chip,
    }


# ==============================================================================
# 輔助函式：專業修復版產業分類判斷（完全避開 2368 等 PCB 股票誤判）
# ==============================================================================
def classify_industry_fixed(code, name):
    code_str = str(code).upper()
    name_str = str(name)

    # 1. ETF / 兩市大盤指數
    if (
        code_str.startswith("00")
        or "ETF" in name_str
        or "增長" in name_str
        or code_str in ["SPCX", "SOXX", "SMH"]
    ):
        return "ETF/大盤指數"

    # 2. 金融保險（嚴格比對：28/58開頭 或 官股/金控特徵，且排除電子股）
    elif (code_str.startswith("28") or code_str.startswith("58")) and (
        "金" in name_str or "銀" in name_str or "保" in name_str
    ):
        return "金融/保險"

    # 3. 航運物流/高鐵/港口
    elif (
        code_str.startswith("26")
        or code_str.startswith("56")
        or "航" in name_str
        or "海" in name_str
        or "宅配" in name_str
        or "高鐵" in name_str
        or "大榮" in name_str
        or "港" in name_str
    ):
        return "航運/物流"

    # 4. 食品/民生/連鎖超商
    elif (
        code_str.startswith("12")
        or code_str.startswith("29")
        or code_str.startswith("59")
        or "超" in name_str
        or "全家" in name_str
    ):
        return "食品/民生消費"

    # 5. 美股科技/半導體/資安/軟體
    elif code_str in [
        "AAPL",
        "META",
        "GOOG",
        "MSFT",
        "NVDA",
        "TSM",
        "ASML",
        "AMAT",
        "LRCX",
        "KLAC",
        "AMD",
        "AVGO",
        "QCOM",
        "INTC",
        "MU",
        "TXN",
        "ARM",
        "MRVL",
        "ADI",
        "MPWR",
        "ON",
        "SWKS",
        "QRVO",
        "TER",
        "MKSI",
        "PANW",
        "CRWD",
        "FTNT",
        "NET",
        "ZS",
        "OKTA",
        "S",
        "GEN",
        "RPD",
        "CBRS",
        "ENTG",
        "SMR",
        "BE",
        "JNJ",
    ]:
        return "美股科技/半導體"

    # 6. 台股電子/半導體/PCB/伺服器/零組件供應鏈
    else:
        return "台股電子/半導體/供應鏈"


# ==============================================================================
# 4. 核心執行、機構級風控與新增優化欄位模組
# ==============================================================================
def run_pipeline():
    print("🚀 開始執行台美股多標的 AI 預測與風控模型流程...\n")

    # 動態取得當前大盤環境狀態與權重
    macro_status, macro_mult = get_macro_market_status()
    print(
        f"📊 當前大盤動態環境判斷：【{macro_status}】 (風控權重係數: {macro_mult})\n"
    )

    results = []

    # 批次下載與運算
    for ticker, name in stock_dict.items():
        df = fetch_stock_data(ticker)
        if df is not None:
            res = train_and_predict(ticker, df)
            if res and res["p_pred"] >= CONFIDENCE_THRESHOLD:
                results.append(
                    {
                        "預測日期": pd.Timestamp.now().strftime("%Y-%m-%d"),
                        "股票代號": ticker,
                        "股票名稱": name,
                        "p_pred_raw": res["p_pred"],
                        "p_hist_raw": res["p_hist"],
                        "ev": res["ev"],
                        "avg_win": res["avg_win"],
                        "avg_loss": res["avg_loss"],
                        "chip_raw": res["chip_ratio"],
                    }
                )

    if not results:
        print("⚠️ 未有標的達到預測信心門檻。")
        return

    df_res = pd.DataFrame(results)

    # --------------------------------------------------------------------------
    # 量化風控計算 (原邏輯完整保留)
    # --------------------------------------------------------------------------
    # (1) 勝率校準 (0.6 ML 預測 + 0.4 歷史勝率)
    df_res["校準後勝率_raw"] = (
        df_res["p_pred_raw"] * 0.6 + df_res["p_hist_raw"] * 0.4
    )

    # (2) 依據「校準後勝率」精準計算半凱利公式 (Half-Kelly Criterion)
    b = df_res["avg_win"] / df_res["avg_loss"]  # 盈虧比
    p_calibrated = df_res["校準後勝率_raw"]
    q_calibrated = 1.0 - p_calibrated

    full_kelly = (b * p_calibrated - q_calibrated) / b
    df_res["kelly_raw"] = np.maximum(0.0, full_kelly / 2.0)

    # (3) 設定動態大盤環境狀態
    df_res["大盤環境狀態"] = macro_status

    # (4) 產業相關性與集中度風控懲罰 (針對大型科技股進行 0.75 懲罰打折)
    tech_tickers = [
        "META",
        "GOOG",
        "AAPL",
        "MSFT",
        "NVDA",
        "TSM",
        "AMD",
        "AVGO",
        "QCOM",
    ]
    df_res["產業相關性懲罰係數"] = df_res["股票代號"].apply(
        lambda s: 0.75 if any(t in s for t in tech_tickers) else 0.95
    )

    # (5) 風控半凱利建議下注
    df_res["風控半凱利建議下注_raw"] = (
        df_res["kelly_raw"] * macro_mult * df_res["產業相關性懲罰係數"]
    )

    # (6) 計算原始部位比率與風控部位比率 (歸一化 Sum to 100%)
    total_orig_kelly = df_res["kelly_raw"].sum()
    total_risk_kelly = df_res["風控半凱利建議下注_raw"].sum()

    df_res["建議部位比率"] = (
        (df_res["kelly_raw"] / total_orig_kelly * 100).round(1).astype(str)
        + "%"
        if total_orig_kelly > 0
        else "0.0%"
    )
    df_res["風控建議部位比率"] = (
        (df_res["風控半凱利建議下注_raw"] / total_risk_kelly * 100)
        .round(1)
        .astype(str)
        + "%"
        if total_risk_kelly > 0
        else "0.0%"
    )

    # --------------------------------------------------------------------------
    # 5. 既有 14 個欄位數值格式化 (完全維持原本定義與名稱)
    # --------------------------------------------------------------------------
    df_res["模型預測漲升機率"] = (df_res["p_pred_raw"] * 100).round(
        2
    ).astype(str) + "%"
    df_res["外資買賣超比"] = (df_res["chip_raw"] * 100).round(2).astype(
        str
    ) + "%"
    df_res["個股歷史勝率"] = (df_res["p_hist_raw"] * 100).round(2).astype(
        str
    ) + "%"
    df_res["單筆期望值(EV)"] = (df_res["ev"] * 100).round(2).astype(
        str
    ) + "%"
    df_res["半凱利建議下注(%)"] = (df_res["kelly_raw"] * 100).round(
        2
    ).astype(str) + "%"
    df_res["校準後勝率"] = (df_res["校準後勝率_raw"] * 100).round(2).astype(
        str
    ) + "%"
    df_res["籌碼與法人動向"] = df_res["chip_raw"].apply(
        lambda x: f"{x*100:+.2f}%"
    )
    df_res["風控半凱利建議下注(%)"] = (
        df_res["風控半凱利建議下注_raw"] * 100
    ).round(2).astype(str) + "%"

    # 原有的 3 個擴充欄位
    is_valid_signal = (df_res["ev"] > 0) & (df_res["校準後勝率_raw"] >= 0.50)
    df_res["訊號有效性"] = np.where(is_valid_signal, "有效交易", "濾除(不建倉)")

    opt_kelly_raw = np.where(
        is_valid_signal, df_res["風控半凱利建議下注_raw"], 0.0
    )
    df_res["優化風控下注(%)"] = (opt_kelly_raw * 100).round(2).astype(str) + "%"

    total_valid_kelly = opt_kelly_raw[is_valid_signal].sum()
    if total_valid_kelly > 0:
        opt_alloc_raw = np.where(
            is_valid_signal, (opt_kelly_raw / total_valid_kelly) * 100.0, 0.0
        )
    else:
        opt_alloc_raw = np.zeros(len(df_res))

    df_res["優化建議部位比率"] = (
        pd.Series(opt_alloc_raw).round(2).astype(str) + "%"
    )

    # --------------------------------------------------------------------------
    # 6. 【專業分析師優化】新增專業擴充欄位
    # --------------------------------------------------------------------------
    # (A) 雙門檻優化： 模型預測漲升機率 >= 50% 且 校準勝率 >= 50% 且 EV > 0%
    strict_valid = (
        (df_res["p_pred_raw"] >= 0.50)
        & (df_res["校準後勝率_raw"] >= 0.50)
        & (df_res["ev"] > 0)
    )
    df_res["專業優化訊號有效性"] = np.where(
        strict_valid, "有效交易", "濾除(不建倉)"
    )

    # (B) 精確修復版產業分類
    df_res["產業分類"] = df_res.apply(
        lambda r: classify_industry_fixed(r["股票代號"], r["股票名稱"]),
        axis=1,
    )

    # (C) 二次優化建議部位比率
    sec_opt_kelly = np.where(strict_valid, df_res["風控半凱利建議下注_raw"], 0.0)
    sec_total = sec_opt_kelly[strict_valid].sum()
    if sec_total > 0:
        sec_alloc_raw = np.where(
            strict_valid, (sec_opt_kelly / sec_total) * 100.0, 0.0
        )
    else:
        sec_alloc_raw = np.zeros(len(df_res))

    df_res["二次優化部位_num"] = sec_alloc_raw

    # (D) 單一產業頂格風控（上限 30%）
    MAX_IND_CAP = 30.0
    ind_sums = df_res.groupby("產業分類")["二次優化部位_num"].transform("sum")

    def apply_cap(row):
        total_ind = ind_sums.loc[row.name]
        raw_val = row["二次優化部位_num"]
        if total_ind > MAX_IND_CAP and total_ind > 0:
            return raw_val * (MAX_IND_CAP / total_ind)
        return raw_val

    df_res["風控頂格建議部位_num"] = df_res.apply(apply_cap, axis=1)

    # (E) 新增專業擴充欄位：動態資金再平衡與現金效率極大化 (Rebalanced Portfolio)
    capped_industries = df_res[ind_sums > MAX_IND_CAP]["產業分類"].unique()
    total_capped_allocated = MAX_IND_CAP * len(capped_industries)

    uncapped_mask = strict_valid & (~df_res["產業分類"].isin(capped_industries))
    remaining_target_pct = 100.0 - total_capped_allocated
    uncapped_current_sum = df_res.loc[uncapped_mask, "二次優化部位_num"].sum()

    if uncapped_current_sum > 0 and remaining_target_pct > 0:
        rebalance_factor = remaining_target_pct / uncapped_current_sum
        df_res["風控頂格再平衡建議部位_num"] = df_res.apply(
            lambda r: (
                r["二次優化部位_num"] * rebalance_factor
                if uncapped_mask.loc[r.name]
                else r["風控頂格建議部位_num"]
            ),
            axis=1,
        )
    else:
        df_res["風控頂格再平衡建議部位_num"] = df_res[
            "風控頂格建議部位_num"
        ]

    # 格式化輸出字串
    df_res["二次優化建議部位比率"] = (
        df_res["二次優化部位_num"].round(2).astype(str) + "%"
    )
    df_res["風控頂格建議部位比率"] = (
        df_res["風控頂格建議部位_num"].round(2).astype(str) + "%"
    )
    df_res["風控頂格再平衡建議部位比率"] = (
        df_res["風控頂格再平衡建議部位_num"].round(2).astype(str) + "%"
    )

    # ==============================================================================
    # 7. 最終輸出欄位排列 (17 個既有欄位 + 4 個專業擴充欄位)
    # ==============================================================================
    final_cols = [
        "預測日期",
        "股票代號",
        "股票名稱",
        "模型預測漲升機率",
        "外資買賣超比",
        "個股歷史勝率",
        "單筆期望值(EV)",
        "半凱利建議下注(%)",
        "建議部位比率",
        "校準後勝率",
        "籌碼與法人動向",
        "大盤環境狀態",
        "風控半凱利建議下注(%)",
        "風控建議部位比率",
        "訊號有效性",
        "優化風控下注(%)",
        "優化建議部位比率",
        # --- 專業分析師擴充優化欄位 ---
        "專業優化訊號有效性",
        "產業分類",
        "二次優化建議部位比率",
        "風控頂格建議部位比率",
        "風控頂格再平衡建議部位比率",
    ]

    output_df = df_res[final_cols]

    # 8. 顯示與匯出結果
    print("✅ 全套 AI 預測與專業機構級風控運算完成！產出結果如下：\n")
    display(output_df)

    output_filename = "recent_5days_confident_with_risk_adjusted_kelly.xlsx"
    output_df.to_excel(output_filename, index=False)
    print(f"\n💾 結果已成功匯出至 Excel 檔案：{output_filename}")

    if HAS_COLAB:
        files.download(output_filename)


# 執行主程式
if __name__ == "__main__":
    run_pipeline()


🚀 開始執行台美股多標的 AI 預測與風控模型流程...

📊 當前大盤動態環境判斷：【強勢多頭】 (風控權重係數: 1.2)

✅ 全套 AI 預測與專業機構級風控運算完成！產出結果如下：



,預測日期,股票代號,股票名稱,模型預測漲升機率,外資買賣超比,個股歷史勝率,單筆期望值(EV),半凱利建議下注(%),建議部位比率,校準後勝率,...,風控半凱利建議下注(%),風控建議部位比率,訊號有效性,優化風控下注(%),優化建議部位比率,專業優化訊號有效性,產業分類,二次優化建議部位比率,風控頂格建議部位比率,風控頂格再平衡建議部位比率
0,2026-09-10,0050.TW,元大台灣50,68.41%,0.0%,57.36%,1.02%,18.53%,1.0%,63.99%,...,21.13%,1.0%,有效交易,21.13%,1.36%,有效交易,ETF/大盤指數,1.42%,1.42%,3.63%
1,2026-09-10,0056.TW,元大高股息,54.12%,0.0%,59.69%,1.28%,13.63%,0.7%,56.35%,...,15.53%,0.7%,有效交易,15.53%,1.0%,有效交易,ETF/大盤指數,1.05%,1.05%,2.67%
2,2026-09-10,00878.TW,國泰永續高股息,81.76%,0.0%,58.2%,1.39%,28.67%,1.6%,72.33%,...,32.69%,1.6%,有效交易,32.69%,2.1%,有效交易,ETF/大盤指數,2.2%,2.2%,5.62%
3,2026-09-10,00981A.TW,統一台股增長主動式,57.78%,0.0%,53.85%,0.32%,5.21%,0.3%,56.21%,...,5.94%,0.3%,有效交易,5.94%,0.38%,有效交易,ETF/大盤指數,0.4%,0.4%,1.02%
4,2026-09-10,AAPL,Apple 蘋果,54.56%,54.57%,49.25%,0.12%,5.03%,0.3%,52.43%,...,4.53%,0.2%,有效交易,4.53%,0.29%,有效交易,美股科技/半導體,0.31%,0.31%,0.78%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,2026-09-10,1216.TW,統一,55.06%,0.0%,56.0%,0.6%,11.91%,0.6%,55.44%,...,13.58%,0.6%,有效交易,13.58%,0.87%,有效交易,食品/民生消費,0.91%,0.91%,2.33%
153,2026-09-10,8182.TWO,加高,53.57%,0.0%,54.0%,-0.5%,0.0%,0.0%,53.74%,...,0.0%,0.0%,濾除(不建倉),0.0%,0.0%,濾除(不建倉),台股電子/半導體/供應鏈,0.0%,0.0%,0.0%
154,2026-09-10,3229.TW,泰藝,49.1%,0.0%,69.77%,8.13%,18.98%,1.0%,57.37%,...,21.64%,1.0%,有效交易,21.64%,1.39%,濾除(不建倉),台股電子/半導體/供應鏈,0.0%,0.0%,0.0%
155,2026-09-10,3308.TW,聯傑,42.17%,0.0%,36.73%,-1.36%,0.0%,0.0%,40.0%,...,0.0%,0.0%,濾除(不建倉),0.0%,0.0%,濾除(不建倉),台股電子/半導體/供應鏈,0.0%,0.0%,0.0%



💾 結果已成功匯出至 Excel 檔案：recent_5days_confident_with_risk_adjusted_kelly.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>